In [1]:
import numpy as np
import matplotlib.pyplot as plt
import stackview as sv
import os
import cv2
from tqdm import tqdm

In [2]:
scan = 'L008'

In [3]:
os.getcwd()

'/home/gabe-j/Desktop/school/current_classes/machine_learning_ii/Machine_learning_II/final_project/medical_imaging_research/notebooks'

In [4]:
raw_dir = '../../raw/'
scans = os.listdir(raw_dir)

In [5]:
def read_raw_image(filepath, dtype, shape = (512, 512)):
    data = np.fromfile(filepath, dtype=dtype)
    return data.reshape(shape)

In [6]:
def normalize_img(img):
    img = img.astype(np.float32)  # ensure float
    img_norm = (img - img.min()) / (img.max() - img.min())
    return img_norm

In [7]:
def read_full_scan(scan, height = 512, width = 512):
    height = height
    width = width
    dtype = np.uint16
    folder = "../../raw/" + scan
    files = os.listdir('../../raw/' + scan)
    files = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
    slices = []

    for f in files:
        if f.endswith(".raw"):
            img = read_raw_image(
                os.path.join(folder, f),
                shape=(height, width),
                dtype=dtype
            )
            img = normalize_img(img)
            slices.append(img)

    volume = np.stack(slices, axis=0)
    return volume

In [8]:
def plot_img(img, figsize=(6,6), title = ''):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title(title)
    plt.show()

In [9]:
import numpy as np
import cv2

def add_bottom_fan_streaks(
    img,
    num_streaks=100,
    intensity=0.05,
    angle_center_deg=0,     # 270° = pointing upward from bottom
    angle_spread_deg=4,      # ± spread
    center_offset_ratio=0.7,   # how far down the center is (0.5=center, ~1=bottom)
    streak_noise_scale = 0.3,
    x_shift = 0.2
):
    img = img.copy().astype(np.float32)
    h, w = img.shape

    # Shift center toward bottom
    cx = 0
    cy = int(h * center_offset_ratio)

    streak_layer = np.zeros_like(img)
    for i in range(2):
        for _ in range(num_streaks//2):
            if i == 0:
                cx = 0 + x_shift*w
            elif i == 1:
                cx = 512 - x_shift*w
            # angle range (in radians)
            angle_deg = np.random.normal(
                loc=angle_center_deg,
                scale=angle_spread_deg / 3
            )

            angle_deg = np.clip(
                angle_deg,
                angle_center_deg - angle_spread_deg,
                angle_center_deg + angle_spread_deg
            )
            angle = np.deg2rad(angle_deg)

            # Start outside/near edge
            r1 = np.random.uniform(0.8, 1.3) * max(h, w)
            r2 = 0  # toward center

            x1 = int(cx + (-1)**(i) * r1 * np.cos(angle))
            y1 = int(cy + r1 * np.sin(angle))

            x2 = int(cx + r2 * np.cos(angle))
            y2 = int(cy + r2 * np.sin(angle))

            thickness = 1

            cv2.line(streak_layer, (x1, y1), (x2, y2), 1, thickness)


    streak_layer = cv2.GaussianBlur(streak_layer, (3, 3), 0)

    mask = streak_layer > 0

    noise = np.random.normal(0, streak_noise_scale, size=streak_layer.shape)
    streak_layer[mask] += noise[mask]

    streak_layer = np.clip(streak_layer, 0, None)
    img_artifact = img + -1 * intensity * streak_layer

    return np.clip(img_artifact, 0, 1)

In [10]:
def add_streaks_to_volume(volume,
                          intensity=0.05,
                          angle_center_deg=0,
                          angle_spread_deg = 6,
                          center_offset_ratio=0.8, 
                          streak_noise_scale=0.3):
    
    new_slices = []
    num_slices = volume.shape[0]
    for i in range(num_slices):
        new_img = add_bottom_fan_streaks(volume[i,:,:],
                                        num_streaks = max(int(80-(i*2)), 0),
                                        intensity=intensity,
                                        angle_center_deg=angle_center_deg,
                                        angle_spread_deg = angle_spread_deg,
                                        center_offset_ratio=center_offset_ratio, 
                                        streak_noise_scale=streak_noise_scale)
        new_slices.append(new_img)
    new_volume = np.stack(new_slices, axis=0)
    return new_volume


In [11]:
def create_side_by_side(vol_1, vol_2):
    side_by_side = []

    for orig, den in zip(vol_1, vol_2):
        combined = np.concatenate([orig, den], axis=1)  # horizontal stack
        side_by_side.append(combined)

    side_by_side = np.stack(side_by_side, axis=0)
    return side_by_side

In [12]:
def write_manipulated_to_file(volume, scan=scan):
    num = volume.shape[0]
    base_file = '../../altered/' + scan + '/'
    os.makedirs(base_file, exist_ok=True)
    for i in tqdm(range(num)):
        file = '../../altered/' + scan + '/' + f'{scan}_{i+1:03d}.raw'
        volume[i,:,:].tofile(file)

In [13]:
scan = 'L123'
volume = read_full_scan(scan)
streaked = add_streaks_to_volume(volume,
                                 intensity = 0.07,
                                 angle_center_deg=0,
                                 angle_spread_deg=8,
                                 center_offset_ratio=0.7,
                                 streak_noise_scale = 0.3)
combo = create_side_by_side(volume, streaked)
print(scan)
sv.slice(combo, slice_number=0, continuous_update=1)

L123


In [14]:
sv.slice(volume, slice_number=1, continuous_update=1)